# 11.3 · 词向量 / Word Embeddings (Word2Vec)

> **课程定位 / Where this fits**
> 第 3 课，**Part 11 · 经典 NLP**。NLP 表示学习的里程碑。
> Lesson 3, **Part 11 · Classic NLP**. A milestone in NLP representation learning.
>
> BoW/TF-IDF 的词彼此**正交、毫无语义**：模型不知道 `cat` 和 `dog` 比 `cat` 和 `table` 更近。**词向量(word embeddings)** 用**低维稠密**向量表示词，让**语义相近的词在空间里也相近**，甚至支持"国王−男人+女人≈女王"的向量类比。这是从"符号"到"语义"的关键一跃，也是现代 NLP/LLM 的根基。本课讲清**分布假设**，并**从零实现 Word2Vec 的 Skip-gram + 负采样**，亲手训练出有语义的词向量并可视化。
> BoW/TF-IDF words are **orthogonal and semantics-free**: the model can't tell that `cat` is closer to `dog` than to `table`. **Word embeddings** represent words as **low-dimensional dense** vectors so **similar-meaning words are close**, even enabling analogies like "king − man + woman ≈ queen." A key leap from symbols to semantics, and the foundation of modern NLP/LLMs. We cover the **distributional hypothesis** and **implement Word2Vec's Skip-gram + negative sampling from scratch**, training semantic word vectors and visualizing them.
>
> 💼 **实战/面试视角**："分布假设 / Skip-gram vs CBOW / 负采样为什么 / 词类比 / 静态 vs 上下文向量" 高频。
> 💼 **Practical/interview angle:** "distributional hypothesis / Skip-gram vs CBOW / why negative sampling / analogies / static vs contextual" — frequent.

> 📐 **符号约定 / Notation**
> - 中心词/上下文词 —— skip-gram 用中心词预测窗口内的上下文词 / center & context words
> - $D$ —— 词向量维度(如 50~300) / embedding dimension
> - 负样本 —— 随机采的"不在上下文"的词 / negative samples (random non-context words)

> 💡 **面试相关 / Interview-relevant**
> - "分布假设是什么"（出镜率 ★★★★，'观其伴而知其义'）
> - "Skip-gram vs CBOW 的区别"（★★★★★）
> - "负采样解决什么问题"（★★★★★，避免全词表 softmax）
> - "词向量为什么能做类比(king-man+woman)"（★★★★）
> - "Word2Vec/GloVe 与 BERT 词向量的区别(静态 vs 上下文)"（★★★★★）

---

## 学习目标 / Learning Objectives
1. 理解分布假设与稠密向量为何能编码语义。
   Understand the distributional hypothesis and why dense vectors encode semantics.
2. 掌握 **Skip-gram** 与 **负采样** 的原理。
   Master Skip-gram and negative sampling.
3. **从零实现并训练** Word2Vec(Skip-gram)。
   Implement and train Word2Vec (Skip-gram) from scratch.
4. 探索**最近邻**与**词类比**，理解能力与局限。
   Explore nearest neighbors and analogies; understand power and limits.
5. **可视化**词向量的语义聚类。
   Visualize semantic clustering of word vectors.

## 目录 / TOC
1. [分布假设：稠密 vs 稀疏 ⭐](#1)
2. [Skip-gram + 负采样：原理 ⭐](#2)
3. [从零训练 Word2Vec ⭐](#3)
4. [最近邻、类比与可视化 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 分布假设：稠密 vs 稀疏 ⭐ / Distributional Hypothesis: Dense vs Sparse

词向量的根基是**分布假设(distributional hypothesis)**：
The foundation is the **distributional hypothesis**:
> **"观其伴而知其义"**——一个词的含义由它经常出现的**上下文**决定。
> **"You shall know a word by the company it keeps."** A word's meaning is determined by the **contexts** it appears in.

如果 `cat` 和 `dog` 经常出现在相似的上下文里(`the ___ ran`, `my pet ___`)，它们语义就相近。词向量就是把这种"上下文相似性"压成**低维稠密向量**：
If `cat` and `dog` appear in similar contexts (`the ___ ran`, `my pet ___`), they're semantically close. Embeddings compress this "context similarity" into **low-dimensional dense vectors**:
- **稀疏 one-hot/BoW**：维度=词表(几万)，每个词彼此正交，**距离都一样、无语义**。
  **Sparse one-hot/BoW:** dimension = vocab (tens of thousands), all words orthogonal — **equidistant, no semantics**.
- **稠密词向量**：维度低(如 100)，每一维是连续值，**语义相近的词向量也相近**(cos 相似度高)。
  **Dense embeddings:** low dimension (e.g. 100), continuous values, **similar words have similar vectors** (high cosine similarity).

下面直观对比两者。
Let's contrast them.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, re, time
from collections import Counter
import torch, torch.nn as nn, torch.nn.functional as F
sns.set_theme(style="whitegrid")

# one-hot: 任意两个不同词的余弦相似度都是 0(完全正交) / one-hot: any two different words have cosine 0
def onehot(i, V): v = np.zeros(V); v[i] = 1; return v
V = 5
print("one-hot 下任意两个不同词的余弦相似度:")
print(f"  cos(词0, 词1) = {onehot(0,V) @ onehot(1,V):.1f}")
print(f"  cos(词0, 词3) = {onehot(0,V) @ onehot(3,V):.1f}")
print("  → 全是0: one-hot 认为所有词两两'一样远', 无法表达 cat 比 table 更像 dog\n")
print("稠密词向量的目标: 让 cat 和 dog 的向量靠近(cos大), cat 和 table 远离(cos小)")
print("做法: 从大量文本中学习 → 上下文相似的词, 向量也相似(分布假设)")


<a id="2"></a>
## 2. Skip-gram + 负采样：原理 ⭐ / Skip-gram + Negative Sampling

**Word2Vec** 有两种训练方式：
**Word2Vec** has two training schemes:
- **CBOW**：用**上下文词**预测**中心词**(多个词→一个词)。
  **CBOW:** predict the **center word** from its **context** (many→one).
- **Skip-gram**：用**中心词**预测**上下文词**(一个词→多个词)。对**罕见词**效果更好，最常用。本课实现它。
  **Skip-gram:** predict **context words** from the **center word** (one→many). Better for **rare words**, most popular. We implement it.

**核心想法**：滑动一个窗口，对每个(中心词, 上下文词)对，让它们的向量**点积大**(相似)。
**Core idea:** slide a window; for each (center, context) pair, make their vectors' **dot product large** (similar).

**问题**：要让"中心词预测正确上下文词"，标准做法是对**整个词表做 softmax**——几万类，每步都要算，太慢。
**Problem:** predicting the right context word via **softmax over the whole vocabulary** (tens of thousands of classes) every step is too slow.

**负采样(negative sampling)** 是关键加速(面试常考)：不做全词表 softmax，而是把问题变成**一堆二分类**——对真实的(中心,上下文)对输出"是"(正样本)，再随机采 $K$ 个**不在上下文的词**作为"否"(负样本)。只需更新 $1+K$ 个词的向量，极快。损失：
**Negative sampling** is the key speedup (interview favorite): instead of full softmax, turn it into **binary classifications** — the true (center, context) pair is "yes" (positive), and $K$ randomly sampled **non-context words** are "no" (negatives). Only $1+K$ vectors update per step — very fast. Loss:

$$-\log \sigma(v_c \cdot v_o) - \sum_{k=1}^{K} \log \sigma(-v_c \cdot v_{neg_k})$$

即：拉高正样本对的相似度、压低负样本对的相似度。
i.e. raise the positive pair's similarity, lower the negatives'.


In [ ]:
import nltk
nltk.download("movie_reviews", quiet=True)
from nltk.corpus import movie_reviews

# 语料: 影评(约百万词), 取前 40 万词训练 / corpus: movie reviews, first 400k tokens
words = [w.lower() for w in movie_reviews.words() if re.match(r"[a-z]+$", w.lower())][:400000]
cnt = Counter(words)
V = 4000                                                  # 只保留最高频的 4000 词 / keep top-4000 words
vocab = [w for w, _ in cnt.most_common(V)]
w2i = {w: i for i, w in enumerate(vocab)}
ids = np.array([w2i[w] for w in words if w in w2i])       # 文本转成词 id 序列 / text → token-id sequence
print(f"语料 {len(words)} 词 → 保留 {len(ids)} 词(在词表内), 词表大小 {len(vocab)}")

# 构造 skip-gram 训练对: 窗口内 (中心词, 上下文词) / build (center, context) pairs within a window
window = 2; centers, contexts = [], []
for off in range(1, window+1):
    centers.append(ids[off:]);  contexts.append(ids[:-off])   # 中心词 → 右侧上下文 / center → right context
    centers.append(ids[:-off]); contexts.append(ids[off:])    # 中心词 → 左侧上下文 / center → left context
C = torch.tensor(np.concatenate(centers)); O = torch.tensor(np.concatenate(contexts))
print(f"训练对(中心,上下文): {len(C):,}")
# 负采样分布: 按词频^0.75(经验值, 既不太偏向高频也照顾低频) / negative-sampling distribution
neg_prob = torch.tensor((np.array([cnt[w] for w in vocab], dtype=float)**0.75), dtype=torch.float)
neg_prob /= neg_prob.sum()
print(f"负采样分布: 按词频^0.75 (Word2Vec 经验设置)")


<a id="3"></a>
## 3. 从零训练 Word2Vec ⭐ / Training Word2Vec From Scratch

实现：两套词向量表——`in_embed`(中心词向量)和 `out_embed`(上下文词向量)。训练时用上面的负采样损失。训练完用 `in_embed` 作为最终词向量。
Implementation: two embedding tables — `in_embed` (center vectors) and `out_embed` (context vectors). Train with the negative-sampling loss above. Use `in_embed` as the final word vectors.


In [ ]:
D = 64; K = 5                                            # 词向量维度; 每个正样本配 5 个负样本 / dim; 5 negatives
torch.manual_seed(0)
in_embed = nn.Embedding(V, D); out_embed = nn.Embedding(V, D)
nn.init.uniform_(in_embed.weight, -0.5/D, 0.5/D); nn.init.zeros_(out_embed.weight)
opt = torch.optim.Adam(list(in_embed.parameters())+list(out_embed.parameters()), lr=5e-3)

B = 4096; n = len(C); t0 = time.time(); loss_hist = []
for epoch in range(3):                                    # 3 轮 / 3 epochs
    perm = torch.randperm(n)
    ep_loss = []
    for i in range(0, n, B):
        idx = perm[i:i+B]; c = C[idx]; o = O[idx]
        neg = torch.multinomial(neg_prob, len(c)*K, replacement=True).view(len(c), K)  # 采负样本 / sample negatives
        vc = in_embed(c); vo = out_embed(o); vn = out_embed(neg)                       # 取向量 / lookup vectors
        pos = F.logsigmoid((vc*vo).sum(1))                # 正样本: 中心·上下文 应大 / positive pair score
        negl = F.logsigmoid(-(vn*vc.unsqueeze(1)).sum(2)).sum(1)   # 负样本: 中心·负样本 应小 / negatives
        loss = -(pos + negl).mean()                       # 负采样损失 / negative-sampling loss
        opt.zero_grad(); loss.backward(); opt.step(); ep_loss.append(loss.item())
    loss_hist.append(np.mean(ep_loss))
print(f"训练完成: {time.time()-t0:.0f}s, 损失 {loss_hist[0]:.2f} → {loss_hist[-1]:.2f}")
E = in_embed.weight.detach().numpy()
E = E / np.linalg.norm(E, axis=1, keepdims=True)          # L2 归一化便于算余弦相似度 / normalize for cosine
print(f"得到 {V} 个词的 {D} 维词向量(稠密!), 对比 one-hot 的 {V} 维稀疏向量")


<a id="4"></a>
## 4. 最近邻、类比与可视化 + 小结 ⭐ / Neighbors, Analogies & Visualization

检验词向量是否学到语义：看一个词的**最近邻**(余弦相似度最高的词)是不是语义相关。
Test whether the vectors learned semantics: a word's **nearest neighbors** (highest cosine similarity) should be semantically related.


In [ ]:
def nearest(word, k=6):
    if word not in w2i: return []
    sims = E @ E[w2i[word]]                               # 与所有词的余弦相似度 / cosine sim to all words
    idx = sims.argsort()[::-1][1:k+1]                     # 取最高的 k 个(排除自己) / top-k excluding itself
    return [vocab[i] for i in idx]

print("最近邻(语义相关的词):")
for w in ["good", "bad", "actor", "music", "war", "funny"]:
    print(f"  {w:8} → {nearest(w)}")
print("\n观察: good→great/nice/decent(近义), actor→actress, music→soundtrack/song → 学到了语义!")
print("(在这个小语料/3秒训练下已有清晰语义结构; 真实 Word2Vec 用数十亿词, 质量更高)")


In [ ]:
# 词类比: king - man + woman ≈ queen 的向量运算 / word analogy via vector arithmetic
def analogy(a, b, c, k=3):
    """a 之于 b, 如同 c 之于 ? → 向量 b - a + c 的最近邻 / b - a + c."""
    if any(w not in w2i for w in [a,b,c]): return ["(词不在词表)"]
    vec = E[w2i[b]] - E[w2i[a]] + E[w2i[c]]              # 类比向量运算 / analogy vector arithmetic
    vec = vec / np.linalg.norm(vec)
    sims = E @ vec
    out = [vocab[i] for i in sims.argsort()[::-1] if vocab[i] not in [a,b,c]][:k]
    return out

print("词类比(向量运算 b - a + c):")
for a,b,c in [("man","actor","woman"), ("bad","worst","good"), ("one","two","first")]:
    print(f"  {a} : {b}  ::  {c} : {analogy(a,b,c)}")
print("\n类比靠'语义方向'在向量空间里平移(如性别方向); 小语料上不总成功, 大语料(GloVe)更稳")
print("💡 经典: king - man + woman ≈ queen (需大语料才稳定复现)")


In [ ]:
# 可视化: 把若干词的向量降到 2D, 看语义聚类 / 2D visualization of semantic clusters
from sklearn.decomposition import PCA
groups = {
    "正面形容词": ["good","great","excellent","wonderful","nice","brilliant"],
    "负面形容词": ["bad","worst","boring","terrible","awful","poor"],
    "电影元素":   ["actor","actress","director","plot","character","scene"],
    "数字":       ["one","two","three","four","five"],
}
sel = [w for g in groups.values() for w in g if w in w2i]
labels_g = [g for g, ws in groups.items() for w in ws if w in w2i]
pts = PCA(n_components=2).fit_transform(E[[w2i[w] for w in sel]])   # PCA 降到 2D / PCA to 2D
fig, ax = plt.subplots(figsize=(9, 6))
palette = {g: c for g, c in zip(groups, ["#2a9d8f","#e76f51","#264653","#e9c46a"])}
for (x, y), w, g in zip(pts, sel, labels_g):
    ax.scatter(x, y, color=palette[g], s=60)
    ax.annotate(w, (x, y), fontsize=9, xytext=(3,3), textcoords="offset points")
for g, c in palette.items(): ax.scatter([], [], color=c, label=g)
ax.legend(); ax.set_title("词向量 PCA: 同语义组的词聚在一起(从零训练得到)")
plt.tight_layout(); plt.show()
print("PCA 显示: 正面词、负面词、电影元素、数字 各自成簇 → 词向量编码了语义")


```
分布假设: 观其伴而知其义; 上下文相似→语义相似→向量相似
稠密 vs 稀疏: one-hot 正交无语义高维; 词向量低维稠密, 语义近则向量近(cos大)
Word2Vec: CBOW(上下文→中心) / Skip-gram(中心→上下文, 罕见词更好, 最常用)
负采样: 不做全词表softmax, 改成正样本'是'+K个负样本'否'的二分类, 极快
训练: in/out 两套向量, 损失=-logσ(正)-Σlogσ(-负); 用 in_embed 作词向量
能力: 最近邻(语义相关) + 类比(king-man+woman≈queen, 大语料更稳)
局限: 静态(一个词一个向量, 不分语境); 一词多义无法区分 → BERT 上下文向量解决(Part 12)
家族: Word2Vec / GloVe(共现矩阵) / FastText(子词, 处理未登录词)
```

### 💡 面试速查 / Interview cheat-sheet
1. **分布假设**: 上下文相似→语义相似; 词向量把它压成稠密向量。
   Distributional hypothesis: similar context → similar meaning → dense vectors.
2. **Skip-gram vs CBOW**: 中心→上下文(罕见词好) vs 上下文→中心。
   Skip-gram vs CBOW: center→context (better for rare) vs context→center.
3. **负采样**: 避免全词表 softmax, 正样本+K负样本二分类, 加速。
   Negative sampling: avoids full softmax; positive + K negatives binary; fast.
4. **类比**: 向量运算 king-man+woman≈queen(语义方向平移)。
   Analogy: vector arithmetic king-man+woman≈queen (semantic directions).
5. **静态 vs 上下文**: Word2Vec 一词一向量; BERT 按语境给词向量(解决一词多义)。
   Static vs contextual: Word2Vec one-vector-per-word; BERT context-dependent (handles polysemy).

### 下一节 / Next
**11.4 文本分类**——有了文本表示(TF-IDF / 词向量)，就能做最常见的 NLP 任务：判断一段文本属于哪一类(垃圾邮件、新闻分类、意图识别)。我们会用 TF-IDF + 逻辑回归/朴素贝叶斯搭建强基线，并做误差分析。
**11.4 Text Classification** — with text representations (TF-IDF / embeddings), we can do the most common NLP task: assign a text to a category (spam, news topic, intent). We'll build strong baselines with TF-IDF + logistic regression/naive Bayes and do error analysis.
